# Late Order Recovery: Agentic Supply-Chain Workflows on the NVIDIA Stack

**Workshop notebook** -- a single, end-to-end example of a model-driven agent
recovering a late customer shipment using structured tool calls, explicit skills,
and sequence-sensitive evaluation.

---
## 1. Introduction

### What this notebook demonstrates

This notebook walks through a **multi-turn agentic workflow** for late-order
recovery in a supply-chain setting. Rather than building a general-purpose agent
platform, we focus on one rich scenario and show concrete patterns for:

- **Structured tool calling** using a Nemotron-style JSON schema
- **Explicit higher-level skills** composed from deterministic tools
- **Fallback parsing** for malformed or partially structured model outputs
- **Sequence-sensitive evaluation** that checks tool-call ordering, not just outcomes
- **Training-oriented patterns** aligned with NeMo RL, Megatron, and ProRL workflows

### Why supply chain?

Supply-chain operations are a strong domain for agentic workflows because:

1. **Decisions depend on ordered information gathering.** You cannot score a
   transfer option without first knowing which alternate DC has stock, and you
   cannot check alternate stock without first confirming the primary source is
   short. This creates natural, machine-checkable sequence dependencies.

2. **Tools are deterministic but composition is not.** Each lookup (inventory,
   ETA, capacity) is a pure function, but the agent must decide *which* tools
   to call and in *what order* based on intermediate results.

3. **Mitigations involve tradeoffs.** Expediting from a supplier is faster but
   costlier; transferring from another DC is cheaper but slower. The agent
   must gather enough data to compare options before recommending.

### Why sequence correctness matters

Most agent evaluations check whether the final answer is correct. But in
supply-chain recovery, *how* the agent arrived at the answer matters:

- Calling `get_transfer_eta` before `find_alternate_inventory` is logically
  invalid -- you cannot estimate a transfer without knowing the source.
- Calling `score_recovery_options` before gathering any options is pointless.
- An agent that guesses the right answer without proper investigation would
  be unsafe in production.

We define and enforce these dependencies explicitly, making the evaluation
**sequence-sensitive** rather than outcome-only.

---
## 2. Scenario Definition

### Business problem

Customer order **SO-10482** for **1,200 units** of **SKU-4090** is at risk of
missing its committed delivery date. The order was placed against distribution
center **DC-WEST-01**, which is the primary source. Shipment tracking shows
the order has not yet shipped and the committed date is approaching.

### Target task

> Determine whether order `SO-10482` can still be fulfilled on time from its
> primary source. If not, recommend the best mitigation action from the
> available options: fulfill from original DC, transfer from an alternate DC,
> expedite from a supplier, partially fulfill, or substitute with a compatible
> SKU.

### Success criteria

A successful agent run must:

1. **Diagnose the risk** -- confirm the order exists, retrieve shipment status,
   and identify that the order is at risk.
2. **Check primary fulfillment** -- verify inventory and capacity at DC-WEST-01.
3. **Explore alternates** -- check at least one alternate DC or supplier path.
4. **Score and recommend** -- compare candidate options and produce a final
   recommendation with rationale.
5. **Respect sequence dependencies** -- never call a downstream tool before its
   prerequisites.
6. **Complete in 5-10 tool calls** -- efficient but thorough.

### Sequence dependencies (machine-checkable)

```
get_order
  -> get_shipment_status
  -> get_inventory(source DC)
  -> get_fulfillment_capacity(source DC)
  -> get_supplier_expedite_options

get_inventory(source DC)
  -> find_alternate_inventory
     -> get_transfer_eta

at least one mitigation path explored
  -> score_recovery_options
     -> recommend_action
```

### Mitigation options the agent should evaluate

| Option | Description |
|--------|-------------|
| Primary fulfill | Ship from DC-WEST-01 if enough stock and capacity |
| DC transfer | Transfer from an alternate DC with available inventory |
| Supplier expedite | Rush order directly from a supplier |
| Partial fulfillment | Ship what is available, backorder the rest |
| SKU substitute | Use a compatible alternate SKU if enabled |

---
## 3. Synthetic Data Model

All scenario data lives in `src/scenario_data.py` as plain Python dicts.
The data is small enough to print in full and inspect cell-by-cell.

Tables:
- **ORDERS** -- customer orders (keyed by order_id)
- **SHIPMENTS** -- shipment status per order
- **INVENTORY** -- on-hand inventory by (sku, dc_id)
- **TRANSFER_LANES** -- routes between DCs with lead times
- **SUPPLIER_OPTIONS** -- supplier expedite options by sku
- **FULFILLMENT_CAPACITY** -- available capacity by (dc_id, date)
- **SUBSTITUTE_SKUS** -- optional substitute SKU mappings

In [ ]:
import sys, pathlib
# Ensure src/ is importable from the notebook
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
# Also allow running from repo root
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [ ]:
from src.scenario_data import (
    ORDERS, SHIPMENTS, INVENTORY, TRANSFER_LANES,
    SUPPLIER_OPTIONS, FULFILLMENT_CAPACITY, SUBSTITUTE_SKUS,
)

print("=== Orders ===")
for oid, o in ORDERS.items():
    print(f"  {oid}: {o['qty']} x {o['sku']} -> {o['source_dc']} (due {o['committed_date']})")

print("\n=== Shipments ===")
for oid, s in SHIPMENTS.items():
    print(f"  {oid}: status={s['status']}, shipped={s['shipped_qty']}")

print("\n=== Inventory ===")
for (sku, dc), inv in INVENTORY.items():
    print(f"  {sku} @ {dc}: on_hand={inv['on_hand']}, reserved={inv['reserved']}, available={inv['available']}")

print("\n=== Transfer Lanes ===")
for (fdc, tdc), lane in TRANSFER_LANES.items():
    print(f"  {fdc} -> {tdc}: {lane['lead_days']} days, ${lane['cost_per_unit']}/unit")

print("\n=== Supplier Expedite Options ===")
for sku, opts in SUPPLIER_OPTIONS.items():
    for opt in opts:
        print(f"  {sku} via {opt['supplier']}: {opt['lead_days']} days, ${opt['cost_per_unit']}/unit (MOQ {opt['moq']})")

print("\n=== Fulfillment Capacity ===")
for (dc, dt), cap in FULFILLMENT_CAPACITY.items():
    print(f"  {dc} on {dt}: {cap['remaining']}/{cap['max_units']} remaining")

print("\n=== Substitute SKUs ===")
for sku, subs in SUBSTITUTE_SKUS.items():
    for s in subs:
        print(f"  {sku} -> {s['substitute_sku']} ({s['compatibility']}): {s['notes']}")

---
## 4. Skill Definitions

Skills are **higher-level behaviors** composed from deterministic tools.
The agent selects a skill; the skill defines which tools to call and in what
order. This separation makes it possible to evaluate *skill selection* and
*tool use* independently.

| Skill | Purpose | Tools used |
|-------|---------|------------|
| `diagnose_order_risk` | Understand current order/shipment state | `get_order`, `get_shipment_status` |
| `assess_primary_fulfillment` | Check if source DC can still fulfill | `get_inventory`, `get_fulfillment_capacity` |
| `evaluate_alternate_recovery_paths` | Explore alternate DCs and suppliers | `find_alternate_inventory`, `get_transfer_eta`, `get_supplier_expedite_options` |
| `synthesize_recommendation` | Score options and recommend action | `score_recovery_options`, `recommend_action` |

### Skills vs. tools

- A **tool** is a single deterministic lookup or computation.
- A **skill** is a reusable behavior pattern that orchestrates multiple tools.
- The agent reasons at the **skill level** ("I should diagnose the order risk
  first") and the execution engine translates that into **tool-level** calls.

This two-layer design lets us:
1. Evaluate whether the agent picked the right skill for the situation.
2. Evaluate whether the tools within each skill were called correctly.
3. Supervise at the skill level for training (coarser, more robust signal).

### Skill transitions

Skills follow a strict diagnostic flow. Each skill has a defined set of
valid successors, enforced by `validate_skill_transition()`:

```
diagnose_order_risk -> assess_primary_fulfillment -> evaluate_alternate_recovery_paths -> synthesize_recommendation
```

This ordering is machine-checkable and feeds directly into the
**sequence correctness** evaluator.

In [ ]:
from src.skills import (
    SKILL_NAMES, SKILL_TOOL_PATTERNS, SKILL_TRANSITIONS, SKILL_ORDER,
    SKILL_REGISTRY, validate_skill_transition,
)

print("=== Skill catalog ===\n")
for skill in SKILL_NAMES:
    tools = SKILL_TOOL_PATTERNS[skill]
    print(f"{skill}:")
    print(f"  tools: {' -> '.join(tools)}")
    transitions = SKILL_TRANSITIONS.get(skill, set())
    print(f"  next:  {transitions if transitions else '(terminal)'}")
    print()

print("=== Transition validation examples ===\n")
# Valid
ok, reason = validate_skill_transition(None, "diagnose_order_risk")
print(f"  (start) -> diagnose_order_risk: valid={ok}")
# Invalid: skipping ahead
ok, reason = validate_skill_transition("diagnose_order_risk", "synthesize_recommendation")
print(f"  diagnose -> synthesize (skip): valid={ok}  ({reason})")

### Running skills step by step

Each skill takes a `SkillContext` that accumulates state across the
diagnostic flow. The context records every tool call made, making the
execution fully inspectable.

In [ ]:
from src.skills import (
    SkillContext, diagnose_order_risk, assess_primary_fulfillment,
    evaluate_alternate_recovery_paths, synthesize_recommendation,
)

# Create a context for the diagnostic flow
ctx = SkillContext(order_id="SO-10482")

# Skill 1: Diagnose the order risk
diagnosis = diagnose_order_risk(ctx)
print("=== Skill 1: diagnose_order_risk ===")
print(f"  at_risk:        {diagnosis.is_at_risk}")
print(f"  reason:         {diagnosis.reason}")
print(f"  days_remaining: {diagnosis.days_remaining}")
print(f"  sku:            {diagnosis.sku}")
print(f"  qty:            {diagnosis.qty}")
print(f"  source_dc:      {diagnosis.source_dc}")
print(f"  tool calls so far: {len(ctx.tool_calls)}")
print()

# Skill 2: Assess primary fulfillment
primary = assess_primary_fulfillment(ctx)
print("=== Skill 2: assess_primary_fulfillment ===")
print(f"  can_fulfill:   {primary.can_fulfill}")
print(f"  available_qty: {primary.available_qty}")
print(f"  shortfall:     {primary.shortfall}")
print(f"  capacity_ok:   {primary.capacity_ok}")
print(f"  tool calls so far: {len(ctx.tool_calls)}")
print()

# Skill 3: Evaluate alternate recovery paths
paths = evaluate_alternate_recovery_paths(ctx)
print("=== Skill 3: evaluate_alternate_recovery_paths ===")
for p in paths:
    flag = "OK" if p.feasible else "NOT FEASIBLE"
    print(f"  [{flag}] {p.path_type} from {p.source}: "
          f"eta={p.eta_days}d, ${p.cost_per_unit}/unit, avail={p.available_qty}")
print(f"  tool calls so far: {len(ctx.tool_calls)}")
print()

# Skill 4: Synthesize recommendation
rec = synthesize_recommendation(ctx)
print("=== Skill 4: synthesize_recommendation ===")
print(f"  action:              {rec.action}")
print(f"  expected_delivery:   {rec.expected_delivery}")
print(f"  meets_committed:     {rec.meets_committed_date}")
print(f"  confidence:          {rec.confidence}")
print(f"  rationale:           {rec.rationale}")
print(f"  total tool calls:    {len(ctx.tool_calls)}")

In [ ]:
# Inspect the full tool call trace recorded in the context
print(f"=== Full tool call trace ({len(ctx.tool_calls)} calls) ===\n")
print(f"Skills executed: {' -> '.join(ctx.skills_executed)}\n")

for i, tc in enumerate(ctx.tool_calls, 1):
    print(f"  {i:2d}. [{tc.skill_name}] {tc.tool_name}({tc.arguments})")

print(f"\n  Total: {len(ctx.tool_calls)} tool calls across {len(ctx.skills_executed)} skills")

---
## 5. Tool Definitions

Each tool is a pure function over the synthetic data tables. Tools are
registered in `TOOL_REGISTRY` so the agent loop can dispatch by name.

| Tool | Inputs | Output | Dependencies |
|------|--------|--------|-------------|
| `get_order` | order_id | Order details | (none) |
| `get_shipment_status` | order_id | Shipment state | get_order |
| `get_inventory` | sku, dc_id | Stock levels | get_order |
| `find_alternate_inventory` | sku, region | Alternate DC stock | get_inventory |
| `get_transfer_eta` | from_dc, to_dc, sku, qty | Transfer estimate | find_alternate_inventory |
| `get_supplier_expedite_options` | sku, qty | Supplier rush options | get_order |
| `get_fulfillment_capacity` | dc_id, date | DC capacity | get_order |
| `score_recovery_options` | options, objective | Ranked options | >= 1 mitigation path |
| `recommend_action` | context | Final recommendation | score_recovery_options |

In [ ]:
from src.tools import TOOL_REGISTRY, TOOL_DEPENDENCIES

print(f"Registered tools: {len(TOOL_REGISTRY)}\n")

print("Tool catalog:")
for name, (fn, params, desc) in TOOL_REGISTRY.items():
    plist = ", ".join(f"{k}: {v}" for k, v in params.items())
    print(f"  {name}({plist})")
    print(f"    {desc}")
    deps = TOOL_DEPENDENCIES.get(name, set())
    print(f"    depends on: {deps if deps else '(none)'}")
    print()

print("--- Example tool calls ---\n")

import json

# 1. Look up the order
result = TOOL_REGISTRY["get_order"][0](order_id="SO-10482")
print("get_order('SO-10482'):")
print(json.dumps(result, indent=2))

# 2. Check primary DC inventory
result = TOOL_REGISTRY["get_inventory"][0](sku="SKU-4090", dc_id="DC-WEST-01")
print(f"\nget_inventory('SKU-4090', 'DC-WEST-01'):")
print(json.dumps(result, indent=2))
print(f"\n  -> Shortfall: {1200 - result['available']} units (need 1200, have {result['available']})")

---
## 6. Structured Tool-Call Schema

The model must emit tool calls in a canonical **Nemotron-style JSON format**.
This section defines the schema, shows validation in action, and demonstrates
how invalid outputs are caught before any tool executes.

### Canonical format

```json
{
  "thought": "optional short reasoning summary",
  "tool_call": {
    "name": "get_inventory",
    "arguments": {
      "sku": "SKU-4090",
      "dc_id": "DC-WEST-01"
    }
  }
}
```

### Final answer format (terminates the loop)

```json
{
  "thought": "reasoning about the recommendation",
  "final_answer": {
    "action": "transfer from DC-EAST-02",
    "rationale": "...",
    "expected_delivery": "2026-04-18",
    "meets_committed_date": true,
    "confidence": 0.85
  }
}
```

### Validation checks

1. Top-level keys must be a subset of `{thought, tool_call, final_answer}`
2. Exactly one of `tool_call` or `final_answer` must be present
3. `tool_call.name` must be a registered tool
4. `tool_call.arguments` must match the tool's expected parameters
5. No extra or unknown argument keys

In [ ]:
from src.schema import (
    validate_tool_call, check_dependencies, extract_json,
    ParsedToolCall, ParsedFinalAnswer, ValidationError,
    REQUIRED_KEYS, OPTIONAL_KEYS, ALLOWED_TOP_KEYS,
)
from src.tools import TOOL_REGISTRY, TOOL_DEPENDENCIES

print("Schema constants:")
print(f"  Required top-level keys: {REQUIRED_KEYS}")
print(f"  Optional top-level keys: {OPTIONAL_KEYS}")
print(f"  Allowed top-level keys:  {ALLOWED_TOP_KEYS}")
print(f"  Registered tools:        {sorted(TOOL_REGISTRY.keys())}")

In [ ]:
# --- Validation examples: valid tool calls ---

print("=== Valid examples ===\n")

# 1. Minimal valid tool call
raw1 = '{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482"}}}'
r1 = validate_tool_call(raw1, TOOL_REGISTRY)
print(f"1. Minimal tool call:  {type(r1).__name__} -> {r1.tool_name}({r1.arguments})")

# 2. With thought field
raw2 = '{"thought": "Start by checking the order", "tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482"}}}'
r2 = validate_tool_call(raw2, TOOL_REGISTRY)
print(f"2. With thought:       {type(r2).__name__} -> thought='{r2.thought}'")

# 3. JSON embedded in surrounding text (common model output)
raw3 = 'Let me check the inventory.\n{"tool_call": {"name": "get_inventory", "arguments": {"sku": "SKU-4090", "dc_id": "DC-WEST-01"}}}\nDone.'
r3 = validate_tool_call(raw3, TOOL_REGISTRY)
print(f"3. Extracted from text: {type(r3).__name__} -> {r3.tool_name}")

# 4. Final answer
raw4 = '{"thought": "Transfer is the best option", "final_answer": {"action": "transfer", "rationale": "lowest cost"}}'
r4 = validate_tool_call(raw4, TOOL_REGISTRY)
print(f"4. Final answer:       {type(r4).__name__} -> {r4.answer}")

In [ ]:
# --- Validation examples: invalid tool calls ---

print("=== Invalid examples ===\n")

# 1. No JSON at all
raw_bad1 = "I think we should check the order first."
r = validate_tool_call(raw_bad1, TOOL_REGISTRY)
print(f"1. No JSON:           {r.error_type}: {r.message}")

# 2. Missing tool_call key (flat structure)
raw_bad2 = '{"name": "get_order", "arguments": {"order_id": "SO-10482"}}'
r = validate_tool_call(raw_bad2, TOOL_REGISTRY)
print(f"2. Missing tool_call: {r.error_type}: {r.message}")

# 3. Unknown tool name
raw_bad3 = '{"tool_call": {"name": "query_database", "arguments": {"sql": "SELECT *"}}}'
r = validate_tool_call(raw_bad3, TOOL_REGISTRY)
print(f"3. Unknown tool:      {r.error_type}: {r.message[:80]}...")

# 4. Missing required arguments
raw_bad4 = '{"tool_call": {"name": "get_inventory", "arguments": {"sku": "SKU-4090"}}}'
r = validate_tool_call(raw_bad4, TOOL_REGISTRY)
print(f"4. Missing args:      {r.error_type}: {r.message}")

# 5. Extra arguments
raw_bad5 = '{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482", "verbose": true}}}'
r = validate_tool_call(raw_bad5, TOOL_REGISTRY)
print(f"5. Extra args:        {r.error_type}: {r.message}")

In [ ]:
# --- Dependency checking ---

print("=== Sequence dependency checks ===\n")

# Valid: get_order has no prerequisites
ok, reason = check_dependencies("get_order", [], TOOL_DEPENDENCIES)
print(f"get_order (no prior calls):           valid={ok}")

# Valid: get_shipment_status after get_order
ok, reason = check_dependencies("get_shipment_status", ["get_order"], TOOL_DEPENDENCIES)
print(f"get_shipment_status after get_order:  valid={ok}")

# Invalid: get_transfer_eta before find_alternate_inventory
ok, reason = check_dependencies("get_transfer_eta", ["get_order", "get_inventory"], TOOL_DEPENDENCIES)
print(f"get_transfer_eta (missing prereq):    valid={ok}")
print(f"  reason: {reason}")

# Invalid: recommend_action before score_recovery_options
ok, reason = check_dependencies("recommend_action", ["get_order", "get_inventory"], TOOL_DEPENDENCIES)
print(f"recommend_action (missing prereq):    valid={ok}")
print(f"  reason: {reason}")

---
## 7. Agent Loop

The agent loop follows the **OpenCode** execution model:

```
think -> emit tool call -> validate -> execute -> observe -> continue
```

Key design choices:

- **Bounded iterations**: hard stop at 15 iterations to prevent runaway loops.
- **Validation before execution**: every tool call is checked against both the
  schema *and* the dependency graph *before* the tool runs.
- **Error feedback**: when the model produces invalid output, the error is fed
  back as a user message so the model can self-correct on the next turn.
- **Full trace capture**: every step is recorded in an `AgentTrace` so the
  notebook can replay and evaluate the trajectory afterward.
- **Single model adapter**: the only network call is `call_model()` in
  `src/agent_loop.py`. Everything else is local and deterministic.

### Architecture

```
                     +------------------+
                     |  System prompt   |
                     | (tools, format)  |
                     +--------+---------+
                              |
                     +--------v---------+
              +----->|  call_model()    |  <- single HTTP adapter
              |      +--------+---------+
              |               |
              |      +--------v---------+
              |      | validate_tool_   |  <- schema + dependency check
              |      | call()           |
              |      +--------+---------+
              |               |
              |      +--------v---------+
              |      |  execute tool    |  <- deterministic lookup
              |      +--------+---------+
              |               |
              |      +--------v---------+
              +------+ append to trace  |  <- full trajectory captured
                     +------------------+
```

### Stop conditions

The loop terminates when:
1. The model emits a **final answer** (`final_answer` key instead of `tool_call`).
2. **Max iterations** reached (safety bound).
3. An **unrecoverable error** occurs (e.g. model endpoint down).

In [ ]:
from src.agent_loop import (
    MODEL_ENDPOINT, MODEL_NAME, MAX_ITERATIONS,
    AgentTrace, ToolCallRecord,
    call_model, build_system_prompt, build_task_message,
    run_agent, print_trace_summary, trace_to_trajectory,
)

print("=== Agent loop configuration ===\n")
print(f"  Model endpoint:  {MODEL_ENDPOINT}")
print(f"  Model name:      {MODEL_NAME}")
print(f"  Max iterations:  {MAX_ITERATIONS}")

# Show the system prompt (truncated)
prompt = build_system_prompt()
print(f"\n=== System prompt ({len(prompt)} chars) ===\n")
print(prompt[:500])
print("...")

### Live agent run

The cell below calls the real Nemotron model to investigate order SO-10482.
The agent will:
1. Receive the task via the system prompt and user message
2. Emit structured tool calls one at a time
3. Have each call validated against the schema and dependency graph
4. Receive tool results as observations
5. Continue until it emits a final answer or hits the iteration limit

**Note**: The model may produce some validation errors along the way.
This is expected with smaller models -- the error feedback loop lets the
model self-correct. Phase 5 will add robust fallback parsing to reduce
these errors further.

In [ ]:
# Run the agent loop against the live model
trace = run_agent("SO-10482", verbose=True, temperature=0.1)

In [ ]:
# Compact trace summary
print_trace_summary(trace)

In [ ]:
# Export the trace as a trajectory (preview for NeMo RL export in Phase 7)
trajectory = trace_to_trajectory(trace)
print(f"Trajectory: {len(trajectory)} steps\n")
for step in trajectory:
    status = "VALID" if step["valid"] else "INVALID"
    print(f"  [{status}] {step['tool_name']}({step['arguments']})")
    if not step["valid"]:
        print(f"           error: {step['validation_error']}")

---
## 8. Fallback Parsing

Real models produce messy outputs. The fallback layer sits between the raw
model output and the schema validator, attempting **mechanical repairs** before
validation. If the output cannot be salvaged, it is **rejected** with a clear
reason.

### Repair-vs-reject policy

| Failure mode | Policy | Rationale |
|---|---|---|
| Trailing commas in JSON | **Repair** | Mechanical fix, intent is clear |
| Single quotes instead of double | **Repair** | Common model error, unambiguous |
| Missing closing braces | **Repair** | Truncated output, can count depth |
| Markdown code fences around JSON | **Repair** | Strip wrapper, extract content |
| Text before/after JSON block | **Repair** | Extract first balanced `{...}` |
| Flat tool call (missing wrapper) | **Repair** | Wrap `{name, arguments}` in `tool_call` |
| Close-match tool name (edit dist <= 2) | **Repair** | e.g., `get_ordr` -> `get_order` |
| Extra unexpected arguments | **Repair** | Remove extras, keep required |
| No JSON object at all | **Reject** | Cannot determine intent |
| Missing `tool_call` and `final_answer` | **Reject** | Ambiguous structure |
| Unknown tool name (no close match) | **Reject** | Could mean anything |
| Missing required arguments | **Reject** | Cannot safely guess values |
| Unsafe argument values (SQL, shell) | **Reject** | Security boundary |

Each repair is tagged with a short label (e.g., `fixed_trailing_commas`,
`corrected_tool_name:get_ordr->get_order`) so the full repair chain is
inspectable in traces.

The `FallbackResult` dataclass records:
- `action`: `REPAIRED`, `REJECTED`, or `NO_ACTION`
- `original`: the raw model output
- `repaired`: the fixed JSON (if repaired)
- `rejection_reason`: why it was rejected (if rejected)
- `repairs_applied`: list of repair labels applied

In [ ]:
from src.fallbacks import try_repair, parse_with_fallback, FallbackAction, FallbackResult
from src.tools import TOOL_REGISTRY

# Helper to display results concisely
def show_fallback(label: str, raw: str, result: FallbackResult) -> None:
    print(f"--- {label} ---")
    print(f"  Input:   {raw[:80]}{'...' if len(raw) > 80 else ''}")
    print(f"  Action:  {result.action.value}")
    if result.repairs_applied:
        print(f"  Repairs: {result.repairs_applied}")
    if result.repaired:
        print(f"  Output:  {result.repaired[:80]}{'...' if len(result.repaired) > 80 else ''}")
    if result.rejection_reason:
        print(f"  Reason:  {result.rejection_reason}")
    print()

### 8a. Repair examples

These outputs are malformed but the intent is clear. The fallback layer
applies mechanical fixes and produces valid tool calls.

In [ ]:
# --- Repair: trailing commas ---
raw_trailing = '{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482",},}}'
show_fallback("Trailing commas", raw_trailing, try_repair(raw_trailing, TOOL_REGISTRY))

# --- Repair: mixed text + JSON ---
raw_mixed = 'Let me check the order details first.\n{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482"}}}\nI will analyze the result next.'
show_fallback("Mixed text + JSON", raw_mixed, try_repair(raw_mixed, TOOL_REGISTRY))

# --- Repair: single quotes ---
raw_single = "{'tool_call': {'name': 'get_order', 'arguments': {'order_id': 'SO-10482'}}}"
show_fallback("Single quotes", raw_single, try_repair(raw_single, TOOL_REGISTRY))

# --- Repair: markdown code fence ---
raw_fenced = '```json\n{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482"}}}\n```'
show_fallback("Markdown fence", raw_fenced, try_repair(raw_fenced, TOOL_REGISTRY))

# --- Repair: truncated JSON (missing closing braces) ---
raw_truncated = '{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482"}'
show_fallback("Truncated JSON", raw_truncated, try_repair(raw_truncated, TOOL_REGISTRY))

# --- Repair: close-match tool name ---
raw_fuzzy = '{"tool_call": {"name": "get_ordr", "arguments": {"order_id": "SO-10482"}}}'
show_fallback("Fuzzy tool name", raw_fuzzy, try_repair(raw_fuzzy, TOOL_REGISTRY))

# --- Repair: flat tool call (missing tool_call wrapper) ---
raw_flat = '{"name": "get_order", "arguments": {"order_id": "SO-10482"}}'
show_fallback("Flat tool call", raw_flat, try_repair(raw_flat, TOOL_REGISTRY))

### 8b. Reject examples

These outputs cannot be salvaged because the intent is ambiguous or the
error is structural. The fallback layer rejects them with a clear reason.

In [ ]:
# --- Reject: no JSON at all ---
raw_no_json = 'I will now check the order status for customer SO-10482.'
show_fallback("No JSON", raw_no_json, try_repair(raw_no_json, TOOL_REGISTRY))

# --- Reject: unknown tool name, no close match ---
raw_unknown = '{"tool_call": {"name": "query_database", "arguments": {"sql": "SELECT *"}}}'
show_fallback("Unknown tool", raw_unknown, try_repair(raw_unknown, TOOL_REGISTRY))

# --- Reject: missing required arguments ---
raw_missing = '{"tool_call": {"name": "get_inventory", "arguments": {"sku": "SKU-4090"}}}'
show_fallback("Missing args", raw_missing, try_repair(raw_missing, TOOL_REGISTRY))

# --- Reject: unsafe argument values ---
raw_unsafe = '{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482; DROP TABLE orders"}}}'
show_fallback("Unsafe args", raw_unsafe, try_repair(raw_unsafe, TOOL_REGISTRY))

# --- Reject: no tool_call or final_answer ---
raw_no_tc = '{"reasoning": "I need to check the order", "next_step": "lookup"}'
show_fallback("No tool_call", raw_no_tc, try_repair(raw_no_tc, TOOL_REGISTRY))

### 8c. Fallback integration in the agent loop

The fallback layer is integrated into the agent loop in `src/agent_loop.py`.
When `validate_tool_call` returns a `ValidationError`, the loop calls
`try_repair` to attempt a mechanical fix. If the repair succeeds,
the repaired output is re-validated and execution continues normally.
If the repair fails, the error is fed back to the model for self-correction.

Each `ToolCallRecord` in the trace now includes:
- `fallback_action`: `"repaired"`, `"rejected"`, or `None`
- `repairs_applied`: list of repair labels (e.g., `["fixed_trailing_commas", "extracted_json_from_text"]`)

The `AgentTrace` also tracks aggregate counts: `fallback_repairs` and
`fallback_rejects`.

```
Model output  -->  validate_tool_call  -->  OK?  -->  Execute tool
                         |                              ^
                         | ValidationError              |
                         v                              |
                    try_repair  -->  REPAIRED?  -->  Re-validate
                         |                              
                         | REJECTED                     
                         v                              
                  Feed error to model (self-correct)
```

### 8d. Worked repair trajectory

This simulates an agent run where the model produces malformed outputs
that the fallback layer repairs before execution. We use synthetic
"model responses" to demonstrate the repair chain without requiring
a live model connection.

In [ ]:
import json
from src.schema import validate_tool_call, check_dependencies
from src.tools import TOOL_REGISTRY, TOOL_DEPENDENCIES
from src.fallbacks import try_repair, FallbackAction

# Simulate a sequence of malformed model outputs and show the repair chain
malformed_outputs = [
    # Step 1: Trailing comma + surrounding text
    'I need to look up the order first.\n'
    '{"thought": "Looking up order details", '
    '"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482",}}}',

    # Step 2: Single-quoted JSON
    "{'thought': 'Check shipment', "
    "'tool_call': {'name': 'get_shipment_status', 'arguments': {'order_id': 'SO-10482'}}}",

    # Step 3: Markdown fence wrapping
    '```json\n'
    '{"tool_call": {"name": "get_inventory", "arguments": {"sku": "SKU-4090", "dc_id": "DC-WEST-01"}}}\n'
    '```',

    # Step 4: Typo in tool name
    '{"tool_call": {"name": "get_fulfilment_capacity", "arguments": {"dc_id": "DC-WEST-01", "date": "2026-04-18"}}}',

    # Step 5: Truncated JSON
    '{"tool_call": {"name": "find_alternate_inventory", "arguments": {"sku": "SKU-4090", "region": "ALL"}',
]

called_tools: list[str] = []
print("=== Worked repair trajectory ===")
print()

for i, raw in enumerate(malformed_outputs, 1):
    print(f"--- Step {i} ---")
    display = raw[:70].replace('\n', '\\n')
    print(f"  Raw:     {display}{'...' if len(raw) > 70 else ''}")

    # First try normal validation
    result = validate_tool_call(raw, TOOL_REGISTRY)

    # If validation fails, try fallback repair
    from src.schema import ValidationError, ParsedToolCall
    if isinstance(result, ValidationError):
        fb = try_repair(raw, TOOL_REGISTRY)
        print(f"  Validation failed: {result.error_type}")
        print(f"  Fallback: {fb.action.value}")
        if fb.repairs_applied:
            print(f"  Repairs:  {fb.repairs_applied}")

        if fb.action == FallbackAction.REPAIRED and fb.repaired:
            result = validate_tool_call(fb.repaired, TOOL_REGISTRY)
            if isinstance(result, ParsedToolCall):
                # Check dependencies
                dep_ok, dep_msg = check_dependencies(result.tool_name, called_tools, TOOL_DEPENDENCIES)
                if dep_ok:
                    fn, _, _ = TOOL_REGISTRY[result.tool_name]
                    tool_result = fn(**result.arguments)
                    called_tools.append(result.tool_name)
                    status_key = next((k for k in ('status', 'available', 'remaining', 'total_available') if k in tool_result), None)
                    preview = f"{status_key}={tool_result[status_key]}" if status_key else 'OK'
                    print(f"  Executed: {result.tool_name} -> {preview}")
                else:
                    print(f"  Dep error: {dep_msg}")
            else:
                print(f"  Still invalid after repair: {result.message if hasattr(result, 'message') else result}")
        else:
            print(f"  Reject:  {fb.rejection_reason}")
    elif isinstance(result, ParsedToolCall):
        dep_ok, _ = check_dependencies(result.tool_name, called_tools, TOOL_DEPENDENCIES)
        if dep_ok:
            fn, _, _ = TOOL_REGISTRY[result.tool_name]
            tool_result = fn(**result.arguments)
            called_tools.append(result.tool_name)
            print(f"  Valid:   {result.tool_name} (no repair needed)")
    print()

print(f"Tools called: {called_tools}")
print(f"All {len(malformed_outputs)} malformed outputs were repaired and executed successfully.")

---
## 9. Worked Examples

This section presents two complete trajectories built from deterministic
tool calls (no live model needed). Both use the same `AgentTrace` structure
that the real agent loop produces, so the evaluators in section 10 can
score them identically.

- **9a. Successful trajectory** -- clean run, correct sequence, valid recommendation.
- **9b. Failure / repair trajectory** -- includes malformed outputs, fallback repairs,
  and a rejected call, showing how the agent recovers.

### 9a. Successful trajectory

A clean run where the agent follows the canonical diagnostic flow:

1. **diagnose_order_risk** -- `get_order` -> `get_shipment_status`
2. **assess_primary_fulfillment** -- `get_inventory` -> `get_fulfillment_capacity`
3. **evaluate_alternate_recovery_paths** -- `find_alternate_inventory` -> `get_transfer_eta` -> `get_supplier_expedite_options`
4. **synthesize_recommendation** -- `score_recovery_options` -> `recommend_action`

Expected: 9 tool calls, all valid, correct sequence, good recommendation.
DC-EAST-02 is the only viable transfer source (DC-CENTRAL-03 cannot
handle the 900-unit shortfall due to its 500-unit lane cap).

In [ ]:
from src.agent_loop import AgentTrace, ToolCallRecord
from src.tools import TOOL_REGISTRY

def build_successful_trace() -> AgentTrace:
    """Build a synthetic successful trajectory for SO-10482.

    Calls tools in the correct dependency order, using real tool
    functions over synthetic data. No model needed.
    """
    trace = AgentTrace(task="Investigate order SO-10482")

    def call(it, name, args, thought):
        fn, _, _ = TOOL_REGISTRY[name]
        result = fn(**args)
        trace.steps.append(ToolCallRecord(
            iteration=it, thought=thought, tool_name=name,
            arguments=args, result=result, valid=True,
        ))
        return result

    # Skill 1: diagnose_order_risk
    call(1, "get_order", {"order_id": "SO-10482"},
         "Start by looking up the order details.")
    call(2, "get_shipment_status", {"order_id": "SO-10482"},
         "Check current shipment status to assess risk.")

    # Skill 2: assess_primary_fulfillment
    call(3, "get_inventory", {"sku": "SKU-4090", "dc_id": "DC-WEST-01"},
         "Check inventory at primary DC.")
    call(4, "get_fulfillment_capacity", {"dc_id": "DC-WEST-01", "date": "2026-04-18"},
         "Check fulfillment capacity at source DC on committed date.")

    # Skill 3: evaluate_alternate_recovery_paths
    call(5, "find_alternate_inventory", {"sku": "SKU-4090", "region": "ALL"},
         "Primary DC has only 300 of 1200 needed. Search all DCs.")
    east_transfer = call(6, "get_transfer_eta",
         {"from_dc": "DC-EAST-02", "to_dc": "DC-WEST-01",
          "sku": "SKU-4090", "qty": 900},
         "DC-EAST-02 has 1000 available. Get transfer ETA for 900-unit shortfall.")
    supplier_opts = call(7, "get_supplier_expedite_options",
         {"sku": "SKU-4090", "qty": 900},
         "Check supplier expedite options for the shortfall.")

    # Skill 4: synthesize_recommendation
    options = [
        {"source": "DC-EAST-02", "description": "dc_transfer from DC-EAST-02",
         "path_type": "dc_transfer", "lead_days": east_transfer["lead_days"],
         "cost_per_unit": east_transfer["cost_per_unit"],
         "total_cost": east_transfer["total_cost"],
         "feasible": east_transfer["feasible"], "covers_full_qty": True},
        {"source": "supplier:GlobalChip Express",
         "description": "supplier_expedite from GlobalChip Express",
         "path_type": "supplier_expedite", "lead_days": 7,
         "cost_per_unit": 8.00, "total_cost": 7200.0,
         "feasible": True, "covers_full_qty": True},
        {"source": "supplier:FastSemi Direct",
         "description": "supplier_expedite from FastSemi Direct",
         "path_type": "supplier_expedite", "lead_days": 5,
         "cost_per_unit": 12.00, "total_cost": 10800.0,
         "feasible": True, "covers_full_qty": True},
    ]
    scored = call(8, "score_recovery_options",
         {"options": options, "objective": "minimize_delay"},
         "Score all recovery options to find the best path.")
    rec = call(9, "recommend_action",
         {"context": {
             "best_option": scored["best_option"],
             "order": {"order_id": "SO-10482", "sku": "SKU-4090",
                       "qty": 1200, "committed_date": "2026-04-18"},
             "objective": "minimize_delay",
         }},
         "Produce the final recommendation based on scored options.")

    trace.final_answer = {
        "action": rec["action"],
        "rationale": rec["rationale"],
        "expected_delivery": rec["expected_delivery"],
        "meets_committed_date": rec["meets_committed_date"],
        "confidence": rec["confidence"],
    }
    trace.completed = True
    trace.stop_reason = "final_answer"
    trace.model_calls = 9
    trace.wall_time_seconds = 8.4  # synthetic
    return trace


success_trace = build_successful_trace()

# Display the trace
from src.agent_loop import print_trace_summary
print_trace_summary(success_trace)

### 9b. Failure / repair trajectory

This trajectory simulates what happens when the model produces malformed
outputs. It includes:

- A **rejected** call (no JSON at all) that the agent retries after error feedback
- A **repaired** call (trailing comma fixed) that executes successfully
- A **repaired** call (typo in tool name fuzzy-matched) that executes successfully

The trajectory still completes with a valid recommendation, but the
extra steps reduce the efficiency score.

In [ ]:
def build_repair_trace() -> AgentTrace:
    """Build a trajectory with fallback repairs and a rejection.

    Demonstrates:
    - Step 2: model emits plain text (rejected), agent retries
    - Step 3: model emits JSON with trailing comma (repaired)
    - Step 6: model emits typo in tool name (repaired via fuzzy match)
    """
    trace = AgentTrace(task="Investigate order SO-10482")

    def call(it, name, args, thought, **kwargs):
        fn, _, _ = TOOL_REGISTRY[name]
        result = fn(**args)
        trace.steps.append(ToolCallRecord(
            iteration=it, thought=thought, tool_name=name,
            arguments=args, result=result, valid=True, **kwargs,
        ))
        return result

    def fail(it, error_msg, fb_action=None, fb_repairs=None):
        trace.steps.append(ToolCallRecord(
            iteration=it, thought=None, tool_name="<invalid>",
            arguments={}, result={"error": error_msg}, valid=False,
            validation_error=error_msg,
            fallback_action=fb_action, repairs_applied=fb_repairs,
        ))

    # Step 1: get_order (clean)
    call(1, "get_order", {"order_id": "SO-10482"},
         "Start by looking up the order details.")

    # Step 2: model emits plain text instead of JSON -> REJECTED
    fail(2, "No JSON object found in model output.",
         fb_action="rejected")
    trace.fallback_rejects += 1

    # Step 3: get_shipment_status (repaired -- trailing comma)
    call(3, "get_shipment_status", {"order_id": "SO-10482"},
         "Check shipping status after retry.",
         fallback_action="repaired",
         repairs_applied=["fixed_trailing_commas"])
    trace.fallback_repairs += 1

    # Step 4: get_inventory (clean)
    call(4, "get_inventory", {"sku": "SKU-4090", "dc_id": "DC-WEST-01"},
         "Check source DC inventory.")

    # Step 5: get_fulfillment_capacity (clean)
    call(5, "get_fulfillment_capacity", {"dc_id": "DC-WEST-01", "date": "2026-04-18"},
         "Check capacity at source DC.")

    # Step 6: find_alternate_inventory (repaired -- tool name typo)
    call(6, "find_alternate_inventory", {"sku": "SKU-4090", "region": "ALL"},
         "Search alternates. Model originally wrote 'find_alternat_inventory'.",
         fallback_action="repaired",
         repairs_applied=["corrected_tool_name:find_alternat_inventory->find_alternate_inventory"])
    trace.fallback_repairs += 1

    # Step 7: get_transfer_eta (clean)
    east_transfer = call(7, "get_transfer_eta",
         {"from_dc": "DC-EAST-02", "to_dc": "DC-WEST-01", "sku": "SKU-4090", "qty": 900},
         "Get transfer ETA from DC-EAST-02.")

    # Step 8: get_supplier_expedite_options (clean)
    supplier_opts = call(8, "get_supplier_expedite_options",
         {"sku": "SKU-4090", "qty": 900},
         "Check supplier expedite for the shortfall.")

    # Step 9: score_recovery_options (clean)
    options = [
        {"source": "DC-EAST-02", "description": "dc_transfer from DC-EAST-02",
         "path_type": "dc_transfer", "lead_days": east_transfer["lead_days"],
         "cost_per_unit": east_transfer["cost_per_unit"],
         "total_cost": east_transfer["total_cost"],
         "feasible": east_transfer["feasible"], "covers_full_qty": True},
        {"source": "supplier:GlobalChip Express",
         "description": "supplier_expedite from GlobalChip Express",
         "path_type": "supplier_expedite", "lead_days": 7,
         "cost_per_unit": 8.00, "total_cost": 7200.0,
         "feasible": True, "covers_full_qty": True},
    ]
    scored = call(9, "score_recovery_options",
         {"options": options, "objective": "minimize_delay"},
         "Score recovery options.")

    # Step 10: recommend_action (clean)
    rec = call(10, "recommend_action",
         {"context": {
             "best_option": scored["best_option"],
             "order": {"order_id": "SO-10482", "sku": "SKU-4090",
                       "qty": 1200, "committed_date": "2026-04-18"},
             "objective": "minimize_delay",
         }},
         "Produce final recommendation.")

    trace.final_answer = {
        "action": rec["action"],
        "rationale": rec["rationale"],
        "expected_delivery": rec["expected_delivery"],
        "meets_committed_date": rec["meets_committed_date"],
        "confidence": rec["confidence"],
    }
    trace.completed = True
    trace.stop_reason = "final_answer"
    trace.model_calls = 12  # includes 2 extra calls for rejected + retries
    trace.wall_time_seconds = 14.2  # synthetic
    return trace


repair_trace = build_repair_trace()
print_trace_summary(repair_trace)

---
## 10. Evaluation

Trajectories are evaluated across **seven dimensions**, each producing
a normalized score from 0.0 to 1.0:

| Dimension | What it measures | Weight |
|-----------|-----------------|--------|
| Skill selection | Did the agent pick the right skills in order? | 15% |
| Tool validity | Were all tool calls well-formed JSON? | 15% |
| Tool accuracy | Were tool arguments correct for the scenario? | 10% |
| Sequence correctness | Were dependency constraints respected? | **25%** |
| Task success | Did the agent reach a valid recommendation? | 15% |
| Recovery quality | If fallback was needed, was repair correct? | 10% |
| Efficiency | Steps taken vs. optimal trajectory length | 10% |

**Sequence correctness** carries the highest weight (25%) because it is
the distinguishing evaluator for this workshop: it checks that tools
were called in an order consistent with the dependency graph, not just
that the final answer was correct.

In [ ]:
from src.evaluation import evaluate_trajectory, print_evaluation

print("=" * 90)
print("EVALUATION: Successful trajectory")
print("=" * 90)
success_eval = evaluate_trajectory(success_trace)
print_evaluation(success_eval)

In [ ]:
print("=" * 90)
print("EVALUATION: Failure / repair trajectory")
print("=" * 90)
repair_eval = evaluate_trajectory(repair_trace)
print_evaluation(repair_eval)

In [ ]:
# Side-by-side comparison
from src.evaluation import EVAL_DIMENSIONS

print(f"{'Dimension':<25} {'Success':>8} {'Repair':>8}  {'Delta':>7}")
print("-" * 55)

s_scores = {s.dimension: s.score for s in success_eval.scores}
r_scores = {s.dimension: s.score for s in repair_eval.scores}

for dim in EVAL_DIMENSIONS:
    s = s_scores[dim]
    r = r_scores[dim]
    delta = r - s
    marker = "" if delta == 0 else ("  ^" if delta > 0 else "  v")
    print(f"{dim:<25} {s:>7.2f}  {r:>7.2f}  {delta:>+6.2f}{marker}")

print("-" * 55)
print(f"{'Overall':<25} {success_eval.overall:>7.2f}  {repair_eval.overall:>7.2f}  "
      f"{repair_eval.overall - success_eval.overall:>+6.2f}")
print()
print(f"Success trajectory: {success_eval.summary}")
print(f"Repair  trajectory: {repair_eval.summary}")

### Sequence correctness deep dive

The cell below demonstrates what happens when the dependency graph
is violated. We construct a trace with a deliberate out-of-order
call (`get_transfer_eta` before `find_alternate_inventory`) and
show how the evaluator catches it.

In [ ]:
# Build a trace with a deliberate sequence violation
bad_trace = AgentTrace(task="Investigate order SO-10482 (bad sequence)")

# Correct start
for name, args in [
    ("get_order", {"order_id": "SO-10482"}),
    ("get_shipment_status", {"order_id": "SO-10482"}),
    ("get_inventory", {"sku": "SKU-4090", "dc_id": "DC-WEST-01"}),
]:
    fn, _, _ = TOOL_REGISTRY[name]
    bad_trace.steps.append(ToolCallRecord(
        iteration=len(bad_trace.steps) + 1, thought="", tool_name=name,
        arguments=args, result=fn(**args), valid=True,
    ))

# VIOLATION: get_transfer_eta BEFORE find_alternate_inventory
fn, _, _ = TOOL_REGISTRY["get_transfer_eta"]
bad_trace.steps.append(ToolCallRecord(
    iteration=4, thought="Jump ahead to transfer", tool_name="get_transfer_eta",
    arguments={"from_dc": "DC-EAST-02", "to_dc": "DC-WEST-01", "sku": "SKU-4090", "qty": 900},
    result=fn(from_dc="DC-EAST-02", to_dc="DC-WEST-01", sku="SKU-4090", qty=900),
    valid=True,
))

bad_trace.completed = False
bad_trace.stop_reason = "max_iterations"
bad_trace.model_calls = 5

from src.evaluation import eval_sequence_correctness
seq_score = eval_sequence_correctness(bad_trace)
print(f"Sequence correctness score: {seq_score.score}")
print(f"Details: {seq_score.details}")
print()
print("The evaluator caught that get_transfer_eta was called before")
print("its prerequisite find_alternate_inventory -- exactly the kind")
print("of sequence error that a final-answer-only evaluator would miss.")

---
## 11. Training-Oriented Discussion

This section connects the workshop patterns to the NVIDIA training stack.

### What should be supervised?

- **Skill level**: which skill the agent selected at each decision point.
  Coarser signal, more robust for training.
- **Trajectory level**: the full sequence of tool calls. Finer signal, but
  noisier because multiple valid orderings may exist.

### How trajectories could be scored

Using the seven evaluation dimensions above as reward components, weighted
by business priority. Sequence correctness should carry the highest weight
to avoid rewarding agents that guess correct answers via invalid paths.

### Sequence-sensitive rewards (ProRL framing)

**NVIDIA ProRL** provides a framework for designing rewards around:
- Tool validity (binary: was the call well-formed?)
- Sequence correctness (dependency graph compliance)
- Recovery quality (did the agent recover from malformed outputs?)
- Task success (did the final recommendation meet criteria?)

### Mapping to NeMo RL / Megatron workflows

- **NeMo RL**: export trajectories with per-step rewards for RL fine-tuning.
- **NVIDIA Megatron**: scale model training on 8x H100 GPUs using the
  trajectory data and reward signals produced above.
- **OpenCode**: the agent loop harness that produced the traces, providing
  the execution and evaluation infrastructure.

### NeMo Guardrails (optional follow-up)

If the fallback parsing layer is extended to include policy checks (e.g.,
rejecting tool calls with unsafe arguments, enforcing business rules),
**NeMo Guardrails** would be the natural integration point. This is noted
as an optional follow-up, not core MVP scope.

In [ ]:
# Placeholder for training-oriented export examples.
# Phase 7 will add concrete NeMo RL export and ProRL reward examples.
print("Training-oriented examples will be populated in Phase 7.")

---
## Next steps

Phases 1-6 have built the complete workshop flow: scenario, data,
tools, skills, schema validation, a live model-driven agent loop,
fallback parsing, worked examples, and seven-dimension evaluation.

The remaining phase will:

- **Phase 7**: Training-oriented discussion -- NeMo RL export, ProRL
  reward design, Megatron alignment, and optional NeMo Guardrails follow-up